# Ty France 2026: Breakout vs Career

2026 season (through 8/14) vs his 2019–2025 career. Run top to bottom from this folder.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pybaseball import cache, statcast_batter

cache.enable()
FIG = "../figures"
PID = 664034                      # Ty France, MLBAM
STATCAST_END = "2026-08-12"       # Statcast pull cutoff
RED, GRAY, LINE = "#D22D49", "#B0AFAF", "#D9D9D9"
INK, MUTED = "#222222", "#888888"
plt.rcParams.update({"font.family": "DejaVu Sans"})

## 1. FanGraphs season stats

Career average = PA-weighted mean of 2019–2025.

In [2]:
# FanGraphs season-by-season export, 2019-2026 (2026 through 8/14)
fg = pd.read_csv("ty_france_career_stats_2026.csv")
fg = fg[fg["Season"].astype(str).str.fullmatch(r"\d{4}")].copy()
fg["Season"] = fg["Season"].astype(int)
fg["WAR600"] = fg["WAR"] / fg["PA"] * 600
car, s26 = fg[fg["Season"] <= 2025], fg[fg["Season"] == 2026].iloc[0]

def career_avg(col):
    """Career average = PA-weighted mean of the 2019-2025 seasons."""
    return (car[col] * car["PA"]).sum() / car["PA"].sum()

print(f"2026: {int(s26.PA)} PA | Career (2019-2025): {int(car.PA.sum()):,} PA")
cols = ["wRC+", "wOBA", "ISO", "BarrelPct", "HardHitPct", "EV", "SoftPct", "WAR600"]
print(pd.DataFrame({"Career": [career_avg(c) for c in cols], "2026": [s26[c] for c in cols]}, index=cols).round(3))

2026: 339 PA | Career (2019-2025): 3,309 PA
             Career     2026
wRC+        110.162  140.000
wOBA          0.323    0.374
ISO           0.138    0.230
BarrelPct     6.789   10.600
HardHitPct   39.193   48.500
EV           88.206   92.300
SoftPct      16.362    7.700
WAR600        1.469    4.248


## 2. Statcast pitch data (2019 to 8/12/26)

In [3]:
# Pitch-level Statcast, one pull per season (regular season only)
parts = [statcast_batter(f"{y}-03-01", f"{y}-11-30" if y < 2026 else STATCAST_END, PID) for y in range(2019, 2027)]
sc = pd.concat(parts, ignore_index=True)
sc = sc[sc["game_type"] == "R"]
sc["season"] = pd.to_datetime(sc["game_date"]).dt.year
keep = ["game_date", "season", "game_pk", "at_bat_number", "pitch_number", "pitch_type", "p_throws",
        "description", "events", "woba_value", "woba_denom"]
sc = sc[keep].sort_values(["game_date", "at_bat_number", "pitch_number"])
sc.to_csv("ty_france_statcast_2019_2026.csv", index=False)
print(sc.groupby("season").size().rename("pitches").to_frame().T)

season   2019  2020  2021  2022  2023  2024  2025  2026
pitches   801   614  2396  2223  2429  2030  1877  1359


## 3. wOBA by pitch type

Fastball includes FF and FA. Curveball includes CU and KC.

In [4]:
# wOBA by pitch type = woba_value / woba_denom on PAs ending with that pitch
GROUP = {"FF": "Fastball", "FA": "Fastball", "SI": "Sinker", "SL": "Slider", "ST": "Sweeper",
         "CH": "Changeup", "FC": "Cutter", "CU": "Curveball", "KC": "Curveball"}
sc["group"] = sc["pitch_type"].map(GROUP)
pa = sc[sc["events"].notna() & (sc["woba_denom"] > 0)]

def woba_by_pitch(g):
    t = g.groupby("group").agg(PA=("woba_denom", "sum"), wv=("woba_value", "sum"))
    return t.assign(wOBA=t["wv"] / t["PA"])[["PA", "wOBA"]]

pt = woba_by_pitch(pa[pa["season"] <= 2025]).join(woba_by_pitch(pa[pa["season"] == 2026]), lsuffix="_career", rsuffix="_2026")
pt["usage_2026"] = sc[sc["season"] == 2026]["group"].value_counts() / (sc["season"] == 2026).sum()
pt = pt.sort_values("usage_2026", ascending=False)
print(pt.round(3))

           PA_career  wOBA_career  PA_2026  wOBA_2026  usage_2026
group                                                            
Fastball      1016.0        0.354    101.0      0.302       0.345
Sinker         632.0        0.345     66.0      0.436       0.186
Slider         584.0        0.305     39.0      0.436       0.112
Sweeper        192.0        0.334     36.0      0.329       0.105
Changeup       322.0        0.301     39.0      0.487       0.082
Cutter         215.0        0.314     23.0      0.398       0.069
Curveball      263.0        0.340     22.0      0.382       0.063


## 4. Charts

In [5]:
# Chart: 2026 vs career (dumbbell, one row per stat)
rows = [("wRC+", "wRC+", lambda v: f"{v:.0f}"), ("wOBA", "wOBA", lambda v: f"{v:.3f}"), ("ISO", "ISO", lambda v: f"{v:.3f}"),
        ("Barrel%", "BarrelPct", lambda v: f"{v:.1f}%"), ("Hard-Hit%", "HardHitPct", lambda v: f"{v:.1f}%"),
        ("Avg Exit Velo", "EV", lambda v: f"{v:.1f} mph"), ("WAR / 600 PA", "WAR600", lambda v: f"{v:.1f}")]

fig = plt.figure(figsize=(9, 7.6))
ax = fig.add_axes([0.0, 0.06, 1.0, 0.80]); ax.axis("off")
ax.set_xlim(0, 1); ax.set_ylim(len(rows) - 0.5, -0.9)
for i, (label, col, f) in enumerate(rows):
    c, n = career_avg(col), s26[col]
    lo, hi = min(c, n), max(c, n)
    pos = lambda v: 0.44 + 0.42 * (v - lo) / (hi - lo)          # each row on its own scale
    ax.plot([pos(c), pos(n)], [i, i], color=LINE, lw=4, zorder=1, solid_capstyle="round")
    for v, color in [(c, GRAY), (n, RED)]:
        ax.scatter(pos(v), i, s=420, color=color, edgecolor="white", linewidth=2, zorder=3)
        ax.text(pos(v), i - 0.30, f(v), ha="center", va="bottom", fontsize=13, fontweight="bold", color=color)
    ax.text(0.14, i, label, ha="left", va="center", fontsize=15, fontweight="bold", color=INK)
ax.text(0.44, -0.62, "Career Avg (2019–2025)", ha="center", fontsize=12, fontweight="bold", color=GRAY)
ax.text(0.86, -0.62, "2026", ha="center", fontsize=12, fontweight="bold", color=RED)
tax = fig.add_axes([0, 0.945, 1, 0.001]); tax.axis("off")
tax.set_title("Ty France: 2026 vs. Career", fontsize=24, fontweight="bold", color=INK, pad=0)
fig.text(0.5, 0.905, f"2026: {int(s26.PA)} PA   |   Career: {int(car.PA.sum()):,} PA (2019–2025)", ha="center", fontsize=13, color="#555555")
fig.text(0.5, 0.02, "Data: FanGraphs.com, 2019–2026 (through 8/14)", ha="center", fontsize=10, color=MUTED, style="italic")
fig.savefig(f"{FIG}/ty_france_vs_career_2026.png", dpi=200)
plt.close(fig)

In [6]:
# Chart: biggest jumps. Rank = z-score of the 2026 change vs his season-to-season spread (2019-2025)
BROWN_D, GOLD_Y, GOLD_T, TAN = "#2B211C", "#FFC425", "#8B6508", "#8C7B6B"
cands = {"wRC+": ("wRC+", "{:.0f}"), "wOBA": ("wOBA", "{:.3f}"), "ISO": ("ISO", "{:.3f}"), "Barrel%": ("BarrelPct", "{:.1f}%"),
         "Hard-Hit%": ("HardHitPct", "{:.1f}%"), "Avg Exit Velo": ("EV", "{:.1f} mph"), "Soft-Hit%": ("SoftPct", "{:.1f}%"),
         "WAR / 600 PA": ("WAR600", "{:.1f}")}
z = {k: (s26[c] - career_avg(c)) / car[c].std() for k, (c, _) in cands.items()}
top5 = sorted(z, key=lambda k: abs(z[k]), reverse=True)[:5]
print("z-scores:", {k: round(v, 2) for k, v in sorted(z.items(), key=lambda kv: -abs(kv[1]))})

fig = plt.figure(figsize=(9.4, 8))
ax = fig.add_axes([0.0, 0.03, 1.0, 0.74]); ax.axis("off")
ax.set_xlim(0, 1); ax.set_ylim(len(top5) - 0.4, -0.45)
for i, k in enumerate(top5):
    col, fmt = cands[k]
    c, n = career_avg(col), s26[col]
    lo, hi = min(c, n), max(c, n)
    pos = lambda v: 0.38 + 0.45 * (v - lo) / (hi - lo)
    ax.plot([pos(c), pos(n)], [i, i], color="#D9CFC1", lw=4, zorder=1)
    ax.scatter(pos(c), i, s=380, color=BROWN_D, zorder=3)
    ax.scatter(pos(n), i, s=420, color=GOLD_Y, edgecolor=BROWN_D, linewidth=1.5, zorder=3)
    ax.text(pos(c), i - 0.22, fmt.format(c), ha="center", va="bottom", fontsize=12.5, fontweight="bold", color=BROWN_D)
    ax.text(pos(n), i - 0.22, fmt.format(n), ha="center", va="bottom", fontsize=12.5, fontweight="bold", color=GOLD_T)
    pct = (n - c) / c
    ax.text(0.605, i + 0.24, f"{'↑' if pct > 0 else '↓'} {pct:+.0%} change", ha="center", va="center",
            fontsize=13, fontweight="bold", style="italic", color=TAN)
    ax.text(0.06, i, f"#{i + 1}", ha="center", va="center", fontsize=13, fontweight="bold", color="#C4B5A5")
    ax.text(0.098, i, k, ha="left", va="center", fontsize=16, fontweight="bold", color=BROWN_D)
ax.scatter(0.46, -0.78, s=380, color=BROWN_D, clip_on=False)
ax.text(0.485, -0.78, "Career Avg (2019–25)", va="center", fontsize=13, fontweight="bold", color=BROWN_D)
ax.scatter(0.80, -0.78, s=420, color=GOLD_Y, edgecolor=BROWN_D, linewidth=1.5, clip_on=False)
ax.text(0.825, -0.78, "2026", va="center", fontsize=13, fontweight="bold", color=GOLD_T)
tax = fig.add_axes([0, 0.945, 1, 0.001]); tax.axis("off")
tax.set_title("Ty France 2026: The Biggest Jumps", fontsize=23, fontweight="bold", color=BROWN_D, pad=0)
fig.text(0.5, 0.91, "The 5 stats that have changed the most vs. his career norms", ha="center", fontsize=13, style="italic", color="#6E5E50")
fig.text(0.985, 0.012, "Data: FanGraphs.com, 2019–2026", ha="right", fontsize=10, style="italic", color="#A89886")
fig.savefig(f"{FIG}/ty_france_biggest_jumps_2026.png", dpi=200)
plt.close(fig)

z-scores: {'Soft-Hit%': np.float64(-4.39), 'ISO': np.float64(3.61), 'Barrel%': np.float64(3.46), 'Avg Exit Velo': np.float64(3.26), 'Hard-Hit%': np.float64(2.0), 'wOBA': np.float64(1.86), 'WAR / 600 PA': np.float64(1.73), 'wRC+': np.float64(1.47)}


In [7]:
# Chart: wOBA by pitch type, career vs 2026, sorted by 2026 usage (shared wOBA axis)
fig = plt.figure(figsize=(9.2, 8.4))
ax = fig.add_axes([0.0, 0.06, 1.0, 0.82])
ax.axis("off"); ax.set_ylim(len(pt) - 0.5, -0.85)
x0, x1 = 0.10, 0.56                                   # wOBA range across the full width
ax.set_xlim(x0, x1)
for i, (g, r) in enumerate(pt.iterrows()):
    c, n = r["wOBA_career"], r["wOBA_2026"]
    ax.plot([c, n], [i, i], color=LINE, lw=4, zorder=1)
    close = abs(c - n) < 0.012
    for v, color, dy in [(c, GRAY, -0.26 - (0.24 if close else 0)), (n, RED, -0.26)]:
        ax.scatter(v, i, s=380, color=color, edgecolor="white", linewidth=2, zorder=3 if color == RED else 2)
        ax.text(v, i + dy, f"{v:.3f}", ha="center", va="bottom", fontsize=13, fontweight="bold", color=color)
    ax.text(x0 + 0.064, i - 0.08, g, ha="left", va="center", fontsize=16, fontweight="bold", color=INK)
    ax.text(x0 + 0.064, i + 0.2, f"{r.usage_2026:.0%} usage in 2026", ha="left", va="center", fontsize=11, style="italic", color=MUTED)
ax.text(0.345, -0.72, "Career (2019–25)", ha="center", fontsize=11, fontweight="bold", color=GRAY)
ax.text(0.445, -0.72, "2026", ha="center", fontsize=11, fontweight="bold", color=RED)
tax = fig.add_axes([0, 0.955, 1, 0.001]); tax.axis("off")
tax.set_title("Ty France 2026: wOBA by Pitch Type", fontsize=22, fontweight="bold", color=INK, pad=0)
fig.text(0.5, 0.925, "Career (2019–25) vs. 2026 — sorted by 2026 usage", ha="center", fontsize=13, style="italic", color="#555555")
lo_pa, hi_pa = int(pt["PA_2026"].min()), int(pt["PA_2026"].max())
fig.text(0.5, 0.018, f"Small samples on 2026 side ({lo_pa}–{hi_pa} PA per pitch). Data: Baseball Savant via pybaseball, 2019–2026 (through 8/12)",
         ha="center", fontsize=9.5, style="italic", color=MUTED)
fig.savefig(f"{FIG}/ty_france_pitch_type_woba_2026.png", dpi=200)
plt.close(fig)